# FIGARCH: Memoria Longa em Volatilidade

Neste notebook, exploraremos o modelo **FIGARCH** (Fractionally Integrated GARCH),
proposto por **Baillie, Bollerslev & Mikkelsen (1996)**.

O FIGARCH generaliza o modelo GARCH padrao para capturar **dependencia de longo prazo**
na volatilidade, utilizando o operador de **diferenciacao fracionaria** $(1-L)^d$.

**Conteudo:**
1. Memoria longa vs memoria curta
2. O operador de diferenciacao fracionaria $(1-L)^d$
3. Estimando FIGARCH
4. Interpretando o parametro $d$
5. Previsao de volatilidade com memoria longa
6. Exercicios

**Referencias:**
- Baillie, R.T., Bollerslev, T. & Mikkelsen, H.O. (1996). *Fractionally integrated generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 74(1), 3-30.
- Bollerslev, T. (1986). *Generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 31(3), 307-327.
- Engle, R.F. (1982). *Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation*. Econometrica, 50(4), 987-1007.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf
from utils.plot_helpers import plot_long_memory_acf

from archbox.models import FIGARCH, GARCH, IGARCH
from archbox.models.figarch import _fractional_coefficients

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Memoria longa vs memoria curta

Um dos **fatos estilizados** mais importantes em series financeiras e que a funcao de
autocorrelacao (ACF) dos retornos absolutos $|r_t|$ e dos retornos ao quadrado $r_t^2$
decai **hiperbolicamente** (lentamente), e nao exponencialmente como previsto pelo GARCH padrao.

- **Memoria curta** (GARCH): $\text{ACF}(k) \sim c \cdot \rho^k$ — decaimento exponencial
- **Memoria longa** (FIGARCH): $\text{ACF}(k) \sim c \cdot k^{2d-1}$ — decaimento hiperbolico

No grafico log-log, a ACF de um processo com memoria longa aparece como uma **reta**
com inclinacao $2d - 1$.

In [ ]:
# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# Calcular retornos absolutos e ao quadrado
abs_returns = np.abs(returns)
sq_returns = returns ** 2

# Plot ACF dos retornos ao quadrado usando plot_long_memory_acf
fig = plot_long_memory_acf(sq_returns.values, max_lags=200)
plt.suptitle('ACF dos Retornos ao Quadrado', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Plot ACF dos retornos absolutos para comparacao
fig2 = plot_long_memory_acf(abs_returns.values, max_lags=200)
plt.suptitle('ACF dos Retornos Absolutos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Calcular ACF numericamente para analise
acf_sq = acf(sq_returns, nlags=200, fft=True)
print('ACF dos retornos ao quadrado:')
print(f'  Lag 1:   {acf_sq[1]:.4f}')
print(f'  Lag 10:  {acf_sq[10]:.4f}')
print(f'  Lag 50:  {acf_sq[50]:.4f}')
print(f'  Lag 100: {acf_sq[100]:.4f}')
print(f'  Lag 200: {acf_sq[200]:.4f}')
print('\nObserve o decaimento lento — indicativo de memoria longa.')

## 2. O operador de diferenciacao fracionaria $(1-L)^d$

O operador de diferenciacao fracionaria generaliza a diferenciacao inteira:

$$(1-L)^d = \sum_{k=0}^{\infty} \binom{d}{k} (-L)^k = 1 - \sum_{k=1}^{\infty} \delta_k L^k$$

onde os coeficientes $\delta_k$ sao dados recursivamente por:

$$\delta_1 = d, \quad \delta_k = \delta_{k-1} \cdot \frac{k - 1 - d}{k}, \quad k \geq 2$$

Para $0 < d < 1$, os coeficientes $\delta_k > 0$ e decaem hiperbolicamente:
$\delta_k \sim \frac{d}{\Gamma(1-d)} k^{-(1+d)}$ quando $k \to \infty$.

Isso significa que choques passados continuam tendo influencia, mesmo em lags muito distantes.

In [ ]:
# Calcular coeficientes fracionarios para diferentes valores de d
n_lags = 100
d_values = [0.1, 0.4, 0.9]
colors = ['blue', 'green', 'red']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico 1: escala linear
for d, color in zip(d_values, colors, strict=False):
    coeffs = _fractional_coefficients(d=d, n_lags=n_lags)
    axes[0].plot(range(1, n_lags + 1), coeffs, label=f'd = {d}', color=color, alpha=0.8)

axes[0].set_xlabel('Lag k')
axes[0].set_ylabel(r'$\delta_k$')
axes[0].set_title('Coeficientes Fracionarios (Escala Linear)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafico 2: escala log-log (verificar decaimento hiperbolico)
for d, color in zip(d_values, colors, strict=False):
    coeffs = _fractional_coefficients(d=d, n_lags=n_lags)
    lags = np.arange(1, n_lags + 1)
    # Evitar log de valores muito pequenos
    mask = coeffs > 1e-10
    axes[1].plot(np.log(lags[mask]), np.log(coeffs[mask]),
                 label=f'd = {d} (incl. $\\approx$ {-(1+d):.1f})', color=color, alpha=0.8)

axes[1].set_xlabel('log(k)')
axes[1].set_ylabel(r'log($\delta_k$)')
axes[1].set_title('Coeficientes Fracionarios (Escala Log-Log)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Imprimir primeiros coeficientes para cada d
print('Primeiros 10 coeficientes fracionarios:')
print(f'{"k":<5}', end='')
for d in d_values:
    print(f'  d={d:<5}', end='')
print()
print('-' * 35)
for k in range(10):
    print(f'{k+1:<5}', end='')
    for d in d_values:
        coeffs = _fractional_coefficients(d=d, n_lags=10)
        print(f'  {coeffs[k]:<8.5f}', end='')
    print()

print('\nObserve: maior d => coeficientes decaem mais lentamente => mais memoria')

## 3. Estimando FIGARCH

O modelo **FIGARCH(1, d, 1)** e definido por:

$$\sigma_t^2 = \omega + \left[1 - \beta L - \phi L (1-L)^d\right] \epsilon_t^2 + \beta \sigma_{t-1}^2$$

ou equivalentemente:

$$(1 - \beta L) \sigma_t^2 = \omega + \left[1 - \beta L - \phi L (1-L)^d\right] \epsilon_t^2$$

**Parametros:**
- $\omega > 0$: intercepto
- $\phi$: parametro ARCH (analogo ao $\alpha$ no GARCH)
- $d \in (0, 1)$: parametro de diferenciacao fracionaria
- $\beta$: parametro GARCH

**Casos especiais:**
- $d = 0$: GARCH(1,1) padrao
- $d = 1$: IGARCH(1,1)

In [ ]:
# Estimar FIGARCH(1,d,1)
model_fig = FIGARCH(returns.values)
results_fig = model_fig.fit()

# Exibir resultados
print(results_fig.summary())

# Extrair parametros
params = results_fig.params
param_names = results_fig.param_names
print('\n--- Parametros Estimados ---')
for name, val in zip(param_names, params, strict=False):
    print(f'{name:>10}: {val:.6f}')

# Identificar o parametro d
d_idx = param_names.index('d') if 'd' in param_names else 2
d_hat = params[d_idx]
print(f'\nParametro d estimado: {d_hat:.4f}')
print('Interpretacao: ', end='')
if d_hat < 0.5:
    print(f'd = {d_hat:.4f} < 0.5 => memoria longa estacionaria')
elif d_hat < 1.0:
    print(f'd = {d_hat:.4f} >= 0.5 => memoria longa nao-estacionaria')
print(f'AIC: {results_fig.aic:.4f}')

## 4. Interpretando o parametro $d$

O parametro $d$ controla a **memoria** do processo de volatilidade:

| Valor de $d$ | Modelo | Memoria | Decaimento da ACF |
|:---:|:---:|:---:|:---:|
| $d = 0$ | GARCH | Curta | Exponencial |
| $0 < d < 0.5$ | FIGARCH | Longa (estacionario) | Hiperbolico |
| $d = 0.5$ | FIGARCH | Longa (fronteira) | Hiperbolico |
| $0.5 < d < 1$ | FIGARCH | Longa (nao-estacionario) | Hiperbolico |
| $d = 1$ | IGARCH | Unitaria | Nao decai |

Na pratica, valores tipicos de $d$ para indices de acoes estao entre **0.3 e 0.5**,
confirmando que a volatilidade tem memoria longa significativa.

In [ ]:
# Estimar GARCH(1,1) e IGARCH(1,1) para comparacao
model_garch = GARCH(returns.values, p=1, q=1)
res_garch = model_garch.fit()

model_igarch = IGARCH(returns.values)
res_igarch = model_igarch.fit()

# Tabela comparativa
comparison = pd.DataFrame({
    'Modelo': ['GARCH(1,1)', 'FIGARCH(1,d,1)', 'IGARCH(1,1)'],
    'd implicito': [0.0, d_hat, 1.0],
    'AIC': [res_garch.aic, results_fig.aic, res_igarch.aic],
    'BIC': [res_garch.bic, results_fig.bic, res_igarch.bic],
    'Log-Lik': [res_garch.loglike, results_fig.loglike, res_igarch.loglike],
    'Persistencia': [
        res_garch.persistence(),
        f'd={d_hat:.4f}',
        res_igarch.persistence()
    ]
})
print('=== Comparacao: GARCH vs FIGARCH vs IGARCH ===')
print(comparison.to_string(index=False))

# Plot volatilidades condicionais sobrepostas
fig, ax = plt.subplots(figsize=(14, 5))
dates = data.index
ax.plot(dates, res_garch.conditional_volatility, label='GARCH(1,1)', alpha=0.7)
ax.plot(dates, results_fig.conditional_volatility, label=f'FIGARCH (d={d_hat:.3f})', alpha=0.7)
ax.plot(dates, res_igarch.conditional_volatility, label='IGARCH(1,1)', alpha=0.7)
ax.set_title('Volatilidade Condicional: GARCH vs FIGARCH vs IGARCH')
ax.set_xlabel('Data')
ax.set_ylabel('Volatilidade')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nd estimado do FIGARCH: {d_hat:.4f}')
print(f'Melhor modelo por AIC: {comparison.loc[comparison["AIC"].astype(float).idxmin() if comparison["AIC"].dtype != object else comparison["AIC"].idxmin(), "Modelo"]}')

## 5. Previsao de volatilidade com memoria longa

Uma diferenca crucial entre GARCH e FIGARCH esta nas **previsoes de longo prazo**:

- **GARCH**: previsoes convergem **exponencialmente** para a variancia incondicional
- **FIGARCH**: previsoes convergem **hiperbolicamente** (muito mais lentamente)

Isso significa que, apos um choque de volatilidade:
- O GARCH "esquece" rapidamente (meia-vida finita)
- O FIGARCH "lembra" por muito mais tempo

Para horizontes curtos (1-5 dias), ambos os modelos dao previsoes similares.
Para horizontes longos (50-100 dias), as diferencas sao substanciais.

In [ ]:
# Comparar forecasts GARCH vs FIGARCH em horizonte longo
horizon = 100
forecast_garch = res_garch.forecast(horizon=horizon)
forecast_fig = results_fig.forecast(horizon=horizon)

# Volatilidade incondicional do GARCH como referencia
uncond_var = res_garch.unconditional_variance()
uncond_vol = np.sqrt(uncond_var)

# Plot comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Previsao de volatilidade
h = np.arange(1, horizon + 1)
axes[0].plot(h, forecast_garch['volatility'], label='GARCH(1,1)', linewidth=2)
axes[0].plot(h, forecast_fig['volatility'], label=f'FIGARCH (d={d_hat:.3f})', linewidth=2)
axes[0].axhline(uncond_vol, color='gray', linestyle='--', alpha=0.7, label='Vol. Incondicional (GARCH)')
axes[0].set_xlabel('Horizonte (dias)')
axes[0].set_ylabel('Volatilidade Prevista')
axes[0].set_title('Previsao de Volatilidade: GARCH vs FIGARCH')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Subplot 2: Diferenca relativa
diff_pct = (forecast_fig['volatility'] - forecast_garch['volatility']) / forecast_garch['volatility'] * 100
axes[1].plot(h, diff_pct, color='purple', linewidth=2)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Horizonte (dias)')
axes[1].set_ylabel('Diferenca Relativa (%)')
axes[1].set_title('Diferenca FIGARCH vs GARCH (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analise numerica
print('=== Convergencia das Previsoes ===')
for h_check in [1, 5, 10, 25, 50, 100]:
    vol_g = forecast_garch['volatility'][h_check - 1]
    vol_f = forecast_fig['volatility'][h_check - 1]
    print(f'  h={h_check:>3d}: GARCH={vol_g:.6f}  FIGARCH={vol_f:.6f}  diff={abs(vol_f-vol_g)/vol_g*100:.2f}%')

print(f'\nVolatilidade incondicional GARCH: {uncond_vol:.6f}')
print(f'Meia-vida GARCH: {res_garch.half_life():.1f} dias')
print('FIGARCH converge muito mais lentamente (hiperbolicamente).')

## 6. Exercicios

1. **Sensibilidade ao truncamento**: O FIGARCH usa uma expansao truncada do operador
   fracionario. Estime o modelo com `truncation_lag=500` e `truncation_lag=2000`.
   Os resultados mudam significativamente?

2. **Teste de memoria longa**: Compare os AIC de modelos FIGARCH com diferentes
   restricoes sobre $d$. O modelo irrestrito (d estimado) e melhor que $d=0$ (GARCH)
   ou $d=1$ (IGARCH)?

3. **Implicacoes para risco**: Como a memoria longa afeta o calculo de VaR
   para horizontes longos? Compare o VaR de 10 dias usando GARCH vs FIGARCH.

4. **Dados reais**: Aplique o FIGARCH a retornos de diferentes classes de ativos
   (acoes, cambio, commodities). O parametro $d$ varia entre classes?

In [ ]:
# Teste sensibilidade ao truncamento
truncation_lags = [500, 1000, 2000]
results_trunc = {}

for tlag in truncation_lags:
    model = FIGARCH(returns.values, truncation_lag=tlag)
    res = model.fit(disp=False)
    results_trunc[tlag] = res

# Tabela comparativa
rows = []
for tlag in truncation_lags:
    res = results_trunc[tlag]
    p = res.params
    names = res.param_names
    row = {'Truncation Lag': tlag}
    for n, v in zip(names, p, strict=False):
        row[n] = v
    row['AIC'] = res.aic
    row['Log-Lik'] = res.loglike
    rows.append(row)

df_trunc = pd.DataFrame(rows)
print('=== Sensibilidade ao Truncamento ===')
print(df_trunc.to_string(index=False, float_format='%.6f'))

# Tabela comparativa final: GARCH vs FIGARCH vs IGARCH
print('\n=== Tabela Comparativa Final ===')
final_table = pd.DataFrame({
    'Modelo': ['GARCH(1,1)', 'FIGARCH(1,d,1)', 'IGARCH(1,1)'],
    'N. Params': [res_garch.params.shape[0], results_fig.params.shape[0], res_igarch.params.shape[0]],
    'Log-Lik': [res_garch.loglike, results_fig.loglike, res_igarch.loglike],
    'AIC': [res_garch.aic, results_fig.aic, res_igarch.aic],
    'BIC': [res_garch.bic, results_fig.bic, res_igarch.bic],
})
print(final_table.to_string(index=False, float_format='%.4f'))

best_model = final_table.loc[final_table['AIC'].idxmin(), 'Modelo']
print(f'\nMelhor modelo por AIC: {best_model}')
print(f'Melhor modelo por BIC: {final_table.loc[final_table["BIC"].idxmin(), "Modelo"]}')